### 🥇 Pipeline de Modelagem Dimensional - Camada Gold (Ouro)

#### 🎯 Objetivo
Este notebook implementa o processo ETL da **Camada Prata → Camada Gold (Ouro)** utilizando **Star Schema** para análises de negócio otimizadas.

##### 📦 Componentes do Pipeline

**Tabelas Dimensão (6)**
* 👤 **Cliente**: Cadastro de clientes
* 💻 **Produto**: Catálogo de peças automotivas
* 🚗 **Carro**: Modelos e marcas de veículos
* 👥 **Vendedor**: Equipe de vendas
* 🏪 **Loja**: Lojas e grupos
* 📅 **Tempo**: Calendário analítico

**Tabela Fato**
* 🎯 **Vendas**: Transações com métricas calculadas (faturamento, lucro, margem)

**Consultas Analíticas**
* 📊 Análises pré-construídas para insights de negócio

---

#### 🔄 Fluxo de Dados
```
Silver Layer (tb_clean_data) -> Temp View (vw_vendas) -> Dimensões (6 tabelas) → Tabela Fato (tb_fato_vendas) -> Consultas Analíticas
```

---

#### 🏗️ Arquitetura

| Componente | Descrição |
|------------|-----------|
| **Modelo** | Star Schema |
| **Dimensões** | 6 tabelas com surrogate keys (SK) |
| **Técnica de Carga** | MERGE (upsert incremental) |
| **Métricas** | Faturamento, Lucro, Margem calculados |


---

#### 📍 Localização no Data Lakehouse

* **Catalog**: `vendas_pecas`
* **Schema**: `camada_ouro`
* **Formato**: Delta Lake (ACID compliant)

In [0]:
USE vendas_pecas.camada_ouro;

##### 📦 Etapa 0: Preparação da Fonte de Dados

###### 🎯 Objetivo
Criar uma **temp view centralizada** (`vw_vendas`) que serve como **fonte única de dados** (Single Source of Truth) para todas as tabelas dimensão e fato.

###### 🔄 Fluxo de Dados

```
Silver Layer (tb_clean_data) -> Filtro de Qualidade (WHERE IdNotaFiscal IS NOT NULL) -> View Temporária (vw_vendas)
```

#### 📝 Detalhes da Implementação

##### Fonte de Dados
* **Tabela**: `vendas_pecas.camada_prata.tb_clean_data`
* **Camada**: Prata (Silver) - dados limpos e tratados

##### Tipo de View
* **CREATE OR REPLACE TEMP VIEW**: View temporária da sessão
* **Escopo**: Visível apenas durante a execução do notebook
* **Idempotência**: Pode ser recriada múltiplas vezes sem erros

#### 💾 Output

**View Criada**: `vw_vendas`

In [0]:
CREATE OR REPLACE TEMP VIEW vw_vendas
AS
SELECT *
FROM 
  vendas_pecas.camada_prata.tb_clean_data cd
WHERE 
  cd.IdNotaFiscal IS NOT NULL

### 📊 Tabelas Dimensão (Dimension Tables)

#### 📖 Conceito de Modelagem Dimensional

**Dimensões** são tabelas que contêm **atributos descritivos** (qualitativos) do negócio. No modelo **Star Schema**, elas:

* 🎯 Circundam a tabela fato (centro da estrela)
* 📝 Fornecem **contexto** para as métricas quantitativas
* 🔍 Respondem perguntas: **Quem?** **O quê?** **Onde?** **Quando?** **Como?**
* ⚡ Otimizam consultas analíticas (menos JOINs que modelo normalizado)

---

#### 🔧 Padrão de Implementação (3 Etapas)

Cada dimensão segue este padrão consistente de 3 etapas:

---
##### 🟢 Etapa 1: CREATE TABLE (Estrutura)
Cria a estrutura física da dimensão

##### 🟡 Etapa 2: CREATE TEMP VIEW (Extração e Deduplicação)
Extrai dados distintos da fonte:

**Objetivos:**
* ✅ **Deduplicação**: ROW_NUMBER() garante registros únicos
* ✅ **Atualidade**: ORDER BY ingest_date DESC pega versão mais recente

##### 🔵 Etapa 3: MERGE (Carga Incremental)
Carga inteligente com upsert

**Comportamento:**
* 🔄 **MATCHED**: Registro já existe → **Atualiza** atributos
* ➕ **NOT MATCHED**: Registro novo → **Insere** com novo SK
* ♻️ **Idempotência**: Pode executar múltiplas vezes sem duplicar dados

In [0]:
USE vendas_pecas.camada_ouro;

#### 💻 1️⃣ Dimensão de Produtos

##### 🎯 Objetivo
Armazenar o **catálogo de peças automotivas** comercializadas, permitindo análises de desempenho por produto.

##### 📝 Estrutura da Tabela

| Coluna | Tipo | Descrição | Chave |
|--------|------|------------|-------|
| `sk_produto` | BIGINT | Surrogate Key auto-incremental | **PK** |
| `produto_peca` | STRING | Nome da peça (normalizado sem acentuação) | - |

##### 🔧 Normalização
* **Deduplicação**: ROW_NUMBER() por produto para garantir unicidade
* **Ordenação**: Usa registro mais recente (ingest_date, data_venda DESC)
* **Padronização**: Nomes sem acentuação para facilitar buscar e análises

In [0]:
-------- CRIANDO TABELA DIMENSAO -----------

-- CREATE OR REPLACE TABLE tb_dim_produtos (
--   sk_produto BIGINT GENERATED ALWAYS AS IDENTITY,
--   produto_peca STRING  
-- );


------------ TEMP VIEW ------------

CREATE OR REPLACE TEMP VIEW vw_dim_produtos
AS
SELECT 
  produto_peca_sem_acento AS produto_peca
FROM 
  (
    SELECT 
      ROW_NUMBER() OVER(PARTITION BY produto_peca_sem_acento 
                        ORDER BY ingest_date, data_venda  DESC 
                       ) AS rn, 
      produto_peca_sem_acento
    FROM 
      vw_vendas
  ) 
WHERE rn = 1;


-------------- MERGE -------------- 

MERGE INTO tb_dim_produtos dp
USING 
  vw_dim_produtos st
ON dp.produto_peca = st.produto_peca
WHEN 
  MATCHED 
THEN
  UPDATE SET dp.produto_peca = st.produto_peca
WHEN
  NOT MATCHED
THEN
  INSERT (produto_peca) VALUES (st.produto_peca)

       

#### 👤 2️⃣ Dimensão de Clientes

##### 🎯 Objetivo
Armazenar **base de clientes** com identificação única e dados cadastrais para análises de comportamento de compra e segmentação.

##### 📝 Estrutura da Tabela

| Coluna | Tipo | Descrição | Chave |
|--------|------|------------|-------|
| `sk_cliente` | BIGINT | Surrogate Key (chave artificial) | **PK** |
| `cliente_id` | BIGINT | Natural Key (ID original do sistema fonte) | **NK** |
| `cliente_nome` | STRING | Nome completo do cliente | - |


In [0]:
-------- CRIANDO TABELA DIMENSAO -----------

-- CREATE OR REPLACE TABLE tb_dim_cliente (
--   sk_cliente BIGINT GENERATED ALWAYS AS IDENTITY,
--   cliente_id BIGINT,
--   cliente_nome STRING  
-- );


------------ TEMP VIEW ------------

CREATE OR REPLACE TEMP VIEW vw_dim_cliente
AS
SELECT 
  cliente_id,
  cliente_nome
FROM
  (
    SELECT 
      ROW_NUMBER () OVER (PARTITION BY cliente_id
                          ORDER BY ingest_date, data_venda DESC) AS rn,
      cliente_id,
      cliente_nome
    FROM vw_vendas
  )
WHERE rn = 1;


-------------- MERGE -------------- 

MERGE INTO tb_dim_cliente dc
USING 
  vw_dim_cliente st
ON dc.cliente_id = st.cliente_id
WHEN 
  MATCHED 
THEN
  UPDATE SET 
    dc.cliente_id = st.cliente_id, 
    dc.cliente_nome = st.cliente_nome
WHEN
  NOT MATCHED
THEN
  INSERT (cliente_id, cliente_nome) VALUES (st.cliente_id, st.cliente_nome)  
  

#### 🚗 3️⃣ Dimensão de Carros

##### 🎯 Objetivo
Armazenar **catálogo de modelos e marcas** de veículos para análises de compatibilidade de peças e preferências de mercado.

##### 📝 Estrutura da Tabela

| Coluna | Tipo | Descrição | Chave |
|--------|------|------------|-------|
| `sk_carro` | BIGINT | Surrogate Key auto-incremental | **PK** |
| `marca_carro` | STRING | Marca do veículo (ex: Toyota, Ford, Honda) | NK (composta) |
| `modelo_carro` | STRING | Modelo específico (ex: Corolla, Fiesta, Civic) | NK (composta) |

##### ⚠️ Importante
A combinação marca + modelo é única. Um mesmo modelo pode existir em marcas diferentes (ex: Onix Chevrolet vs Onix Plus).

In [0]:
-------- CRIANDO TABELA DIMENSAO -----------

-- CREATE OR REPLACE TABLE tb_dim_carro (
--   sk_carro BIGINT GENERATED ALWAYS AS IDENTITY,
--   marca_carro STRING,
--   modelo_carro STRING
-- );


------------ TEMP VIEW ------------

CREATE OR REPLACE TEMP VIEW vw_dim_carro
AS
SELECT 
  marca_carro,
  modelo_carro
FROM
  (
    SELECT 
      ROW_NUMBER () OVER (PARTITION BY marca_carro, modelo_carro
                          ORDER BY ingest_date, data_venda DESC) AS rn,
      marca_carro,
      modelo_carro
    FROM 
      vw_vendas
  
  )
WHERE rn = 1;


-------------- MERGE -------------- 

MERGE INTO tb_dim_carro dc
USING 
  vw_dim_carro st
ON dc.marca_carro = st.marca_carro
AND dc.modelo_carro = st.modelo_carro
WHEN 
  MATCHED 
THEN
  UPDATE SET 
    dc.marca_carro = st.marca_carro, 
    dc.modelo_carro = st.modelo_carro
WHEN
  NOT MATCHED
THEN
  INSERT (marca_carro, modelo_carro) VALUES (st.marca_carro, st.modelo_carro)
      

#### 👥 4️⃣ Dimensão de Vendedores

##### 🎯 Objetivo
Armazenar **equipe de vendas** para avaliação de desempenho individual, comissionamento e gestão comercial.

##### 📝 Estrutura da Tabela

| Coluna | Tipo | Descrição | Chave |
|--------|------|------------|-------|
| `sk_vendedor` | BIGINT | Surrogate Key auto-incremental | **PK** |
| `vendedor_id` | STRING | Código do vendedor (matrícula/ID funcional) | NK (composta) |
| `vendedor_nome` | STRING | Nome completo do vendedor | NK (composta) |


In [0]:
-------- CRIANDO TABELA DIMENSAO -----------

-- CREATE OR REPLACE TABLE tb_dim_vendedor (
--   sk_vendedor BIGINT GENERATED ALWAYS AS IDENTITY,
--   vendedor_id STRING,
--   vendedor_nome STRING
-- );


------------ TEMP VIEW ------------

CREATE OR REPLACE TEMP VIEW vw_dim_vendedor
AS
SELECT
  vendedor_id,
  vendedor_nome
FROM
  (
    SELECT
      ROW_NUMBER() OVER (PARTITION BY vendedor_id
                         ORDER BY ingest_date, data_venda DESC) AS rn,
      vendedor_id,
      vendedor_nome
    FROM
      vw_vendas
  )
WHERE rn = 1;


-------------- MERGE -------------- 

MERGE INTO tb_dim_vendedor dv
USING 
  vw_dim_vendedor st
ON dv.vendedor_id = st.vendedor_id
AND dv.vendedor_nome = st.vendedor_nome
WHEN 
  MATCHED 
THEN
  UPDATE SET 
    dv.vendedor_id = st.vendedor_id, 
    dv.vendedor_nome = st.vendedor_nome
WHEN
  NOT MATCHED
THEN
  INSERT (vendedor_id, vendedor_nome) VALUES (st.vendedor_id, st.vendedor_nome)
      

## 🏪 5️⃣ Dimensão de Lojas

### 🎯 Objetivo
Armazenar **hierarquia organizacional** de lojas e grupos para análises regionais, comparações entre unidades e consolidação de resultados.

### 📝 Estrutura da Tabela

| Coluna | Tipo | Descrição | Chave |
|--------|------|------------|-------|
| `sk_loja` | BIGINT | Surrogate Key auto-incremental | **PK** |
| `loja_id` | BIGINT | ID original da loja (código da unidade) | NK (composta) |
| `loja_nome` | STRING | Nome da loja (normalizado sem acentuação) | NK (composta) |
| `grupo_loja` | STRING | Grupo/região ao qual a loja pertence | Hierarquia |

In [0]:
-------- CRIANDO TABELA DIMENSAO -----------

-- CREATE OR REPLACE TABLE tb_dim_loja (
--   sk_loja BIGINT GENERATED ALWAYS AS IDENTITY,
--   loja_id BIGINT,
--   loja_nome STRING,
--   grupo_loja STRING
-- );


------------ TEMP VIEW ------------

CREATE OR REPLACE TEMP VIEW vw_dim_loja
AS
SELECT
  loja_id,
  loja_nome,
  grupo_loja
FROM
  (
    SELECT
      ROW_NUMBER() OVER (PARTITION BY loja_id
                         ORDER BY ingest_date, data_venda DESC) AS rn,
      loja_id,
      loja_nome_sem_acento AS loja_nome,
      grupo_loja
    FROM
      vw_vendas
  )
WHERE rn = 1;

-------------- MERGE -------------- 

MERGE INTO tb_dim_loja dl
USING 
  vw_dim_loja st
ON dl.loja_id = st.loja_id
AND dl.loja_nome = st.loja_nome
AND dl.grupo_loja = st.grupo_loja
WHEN 
  MATCHED 
THEN
  UPDATE SET 
    dl.loja_id = st.loja_id, 
    dl.loja_nome = st.loja_nome,
    dl.grupo_loja = st.grupo_loja
WHEN
  NOT MATCHED
THEN
  INSERT (loja_id, loja_nome, grupo_loja) VALUES (st.loja_id, st.loja_nome, st.grupo_loja)
      

#### 📅 6️⃣ Dimensão de Tempo (Calendário Analítico)

##### 🎯 Objetivo
Criar um **calendário analítico** (Time Dimension) com atributos temporais pré-calculados para facilitar análises de séries temporais, sazonalidade e comparações periódicas.

##### 📝 Estrutura da Tabela

| Coluna | Tipo | Descrição | Chave |
|--------|------|------------|-------|
| `sk_data_venda` | BIGINT | Surrogate Key auto-incremental | **PK** |
| `data_venda` | DATE | Data completa da venda (formato DATE) | **NK** |
| `ano_venda` | INT | Ano extraído (ex: 2024, 2025) | Atributo |
| `mes_venda` | INT | Mês numérico (1-12) | Atributo |
| `dia_venda` | INT | Dia do mês (1-31) | Atributo |
| `semestre_venda` | INT | Semestre (1 ou 2) | Atributo |

##### 💾 Fonte de Dados
* **Arquivo**: `/Volumes/vendas_pecas/default/my/utils/DM_TEMPO.csv`
* **Formato**: CSV com delimitador `;`
* **Conteúdo**: Calendário pré-gerado com todos os atributos temporais
* **Carga**: Uma única vez (tabela de referência)

In [0]:
-- CREATE OR REPLACE TABLE tb_dim_tempo 
-- AS
-- SELECT 
--   SK_DATA as sk_data_venda,
--   try_to_date(DATA, 'dd/MM/yyyy HH:mm') as data_venda,
--   ANO AS ano_venda,
--   MES AS mes_venda,
--   DIA AS dia_venda,
--   SEMESTRE AS semestre_venda
-- FROM
--   read_files(
--     '/Volumes/vendas_pecas/default/my/utils/DM_TEMPO.csv',
--     format => 'csv',
--     inferschema=> 'true',
--     header => 'true',
--     delimiter => ';'
--    )

In [0]:
-- -------- CRIANDO TABELA DIMENSAO -----------

-- CREATE OR REPLACE TABLE tb_dim_data_venda (
--   sk_data_venda BIGINT GENERATED ALWAYS AS IDENTITY,
--   data_venda DATE,
--   ano_venda INT,
--   mes_venda INT,
--   dia_venda INT
-- );


-- ------------ TEMP VIEW ------------

-- CREATE OR REPLACE TEMP VIEW vw_dim_data_venda
-- AS
-- SELECT
--   data_venda,
--   ano_venda,
--   mes_venda,
--   dia_venda
-- FROM
--   (
--     SELECT
--       ROW_NUMBER() OVER (PARTITION BY data_venda
--                         ORDER BY ingest_date, data_venda DESC) AS rn,
--       data_venda,
--       ano_venda,
--       mes_venda,
--       DAY(data_venda) AS dia_venda
--     FROM
--       vw_vendas
--   )
-- WHERE rn = 1;


-- -------------- MERGE -------------- 

-- MERGE INTO tb_dim_data_venda ddv
-- USING 
--   vw_dim_data_venda st
-- ON ddv.data_venda = st.data_venda
-- AND ddv.ano_venda = st.ano_venda
-- AND ddv.mes_venda = st.mes_venda
-- WHEN 
--   MATCHED 
-- THEN
--   UPDATE SET 
--     ddv.data_venda = st.data_venda, 
--     ddv.ano_venda = st.ano_venda,
--     ddv.mes_venda = st.mes_venda,
--     ddv.dia_venda = st.dia_venda
-- WHEN
--   NOT MATCHED
-- THEN
--   INSERT (data_venda, ano_venda, mes_venda, dia_venda) VALUES (st.data_venda, st.ano_venda, st.mes_venda, st.dia_venda)


##### 🎯 Tabela Fato (Fact Table)

###### 📖 Conceito de Tabela Fato

A **Tabela Fato** é o **centro (coração) do Star Schema**, armazenando:

* 📊 **Métricas Mensuráveis** (Fatos): Valores quantitativos do negócio
* 🔗 **Foreign Keys**: Referências para todas as dimensões (SKs)
* 📅 **Eventos de Negócio**: Cada linha = 1 transação/evento

##### Anatomia de uma Tabela Fato

```
           DIMENSÃO CLIENTE (👤)
                  |
                  | sk_cliente
                  |
   DIMENSÃO       |        DIMENSÃO
   PRODUTO -------|------- DATA
   (💻)         |        (📅)
      sk_produto  |  sk_data_venda
                  |
        +---------+---------+
        |   TABELA FATO    |  ← Centro da Estrela
        |   (tb_fato_      |
        |    vendas)       |
        +---------+---------+
                  |
      sk_loja     |  sk_vendedor
                  |
   DIMENSÃO       |        DIMENSÃO
   LOJA ----------|------- VENDEDOR
   (🏪)          |        (👥)
                  |
                  | sk_carro
                  |
           DIMENSÃO CARRO (🚗)
```

---


#### 📊 Tipos de Fatos

##### 1️⃣ Fatos Aditivos (Fully Additive)
**Podem ser somados em TODAS as dimensões**

| Métrica | Tipo | Descrição | Fórmula |
|---------|------|------------|----------|
| `valor_unitario` | DECIMAL(18,2) | Preço de venda unitário | Fonte |
| `quantidade` | DOUBLE | Quantidade de peças vendidas | Fonte |
| `custo_unitario` | DECIMAL(18,2) | Custo unitário da peça | Fonte |
| `faturamento` | DECIMAL(18,2) | Receita da venda | **valor_unitario × quantidade** |
| `lucro` | DECIMAL(18,2) | Lucro líquido | **faturamento - (custo_unitario × quantidade)** |


##### 2️⃣ Metadados de Auditoria
**Para rastreabilidade e governança**

| Coluna | Tipo | Propósito |
|--------|------|----------|
| `file_path` | STRING | Caminho do arquivo fonte (origem dos dados) |
| `ingest_date` | TIMESTAMP | Data/hora da carga (quando o dado entrou no lakehouse) |

---

##### Por que LEFT JOIN?
* ✅ **Preserva vendas** mesmo se alguma dimensão não for encontrada
* ✅ **Permite análise de dados incompletos** (ex: vendas sem carro associado)
* ✅ **Evita perda de transações** por problemas de qualidade em dimensões

In [0]:
USE vendas_pecas.camada_ouro;

#### 🎯 Implementação: Tabela Fato Vendas

##### 🎯 Objetivo
Criar a **tabela central do Data Warehouse** com todas as transações de venda enriquecidas com:
* 🔗 Referências dimensionais (Foreign Keys)
* 📊 Métricas pré-calculadas (faturamento, lucro)
* 📋 Metadados de auditoria

---

#### 🔧 Processo de Construção (2 Etapas)

 🟢 Etapa 1: CREATE TEMP VIEW (Preparação)


 🔵 Etapa 2: MERGE INTO (Carga Incremental)


In [0]:
------------- VIEW --------------

CREATE OR REPLACE TEMP VIEW vw_fato_vendas AS
SELECT
  fv.IdNotaFiscal,
  dt.sk_data_venda,
  dcl.sk_cliente,
  dc.sk_carro,
  dl.sk_loja,
  dpd.sk_produto,
  dv.sk_vendedor,
  fv.valor_unitario,
  fv.quantidade,
  fv.custo_unitario,
  cast((fv.valor_unitario * fv.quantidade) as decimal (18,2)) as faturamento,
  cast((faturamento - (fv.custo_unitario * fv.quantidade)) as decimal(18,2)) as lucro,
  file_path,
  ingest_date
FROM vw_vendas fv
LEFT JOIN tb_dim_cliente dcl ON fv.cliente_id = dcl.cliente_id
LEFT JOIN tb_dim_tempo dt ON fv.data_venda = dt.data_venda
LEFT JOIN tb_dim_loja dl ON fv.loja_nome_sem_acento = dl.loja_nome
LEFT JOIN tb_dim_produtos dpd ON fv.produto_peca_sem_acento = dpd.produto_peca
LEFT JOIN tb_dim_vendedor dv ON fv.vendedor_nome = dv.vendedor_nome
LEFT JOIN tb_dim_carro dc ON fv.modelo_carro = dc.modelo_carro and fv.marca_carro = dc.marca_carro
ORDER BY IdNotaFiscal;


-------------- MERGE INTO -----------------

MERGE INTO tb_fato_vendas fv
USING
  vw_fato_vendas
ON
  fv.IdNotaFiscal = vw_fato_vendas.IdNotaFiscal
WHEN 
  MATCHED
THEN
  UPDATE SET *
WHEN
  NOT MATCHED
THEN
 INSERT *                                    

#### 📊 Consultas Analíticas

##### Objetivo
Conjunto de **consultas pré-construídas** que respondem perguntas estratégicas de negócio, aproveitando o modelo Star Schema para queries otimizadas.

##### Filtro de Qualidade
Todas as consultas aplicam:
```sql
WHERE
  quantidade IS NOT NULL AND quantidade > 0
  AND valor_unitario > 0
  AND custo_unitario > 0
```
Garante que apenas vendas válidas sejam analisadas.

##### Questões de Negócio
1. 📈 **Evolução Temporal**: Como estão as tendências ao longo do tempo?
2. 📅 **Sazonalidade**: Quais períodos tem melhor/pior performance?
3. 🏪 **Performance por Loja**: Qual o desempenho de lojas e grupos?
4. 🔁 **Comparação Anual**: Como 2024 vs 2025?
5. 📊 **Volume de Vendas**: Onde está concentrado o volume?

In [0]:
USE vendas_pecas.camada_ouro;

### 📈 Questão 1: Evolução Temporal

#### Como evoluíram nosso faturamento e resultados ao longo do tempo?

**Análise Mês a Mês:**
* Faturamento mensal
* Lucro mensal
* Margem de lucro (%)
* Quantidade de peças vendidas

In [0]:
SELECT
  ano_venda AS ano,
  mes_venda AS mes,
  SUM(faturamento) AS faturamento,
  SUM(lucro) AS lucro,
  SUM(lucro) / SUM(faturamento) * 100 AS margem_lucro,
  SUM(quantidade) AS quantidade_pecas
FROM 
  tb_fato_vendas fv
JOIN
  tb_dim_tempo dv ON fv.sk_data_venda = dv.sk_data_venda
WHERE
  fv.quantidade > 0 and
  fv.valor_unitario > 0 and
  fv.custo_unitario > 0 and
  ano_venda BETWEEN 2024 and 2025
GROUP BY 
  mes, ano
ORDER BY 
  ano_venda, mes_venda

In [0]:
SELECT
  ano_venda AS ano,
  mes_venda AS mes,
  SUM(faturamento) AS faturamento,
  SUM(lucro) AS lucro,
  SUM(lucro) / SUM(faturamento) * 100 AS margem_lucro,
  SUM(quantidade) AS quantidade_pecas
FROM 
  tb_bench
WHERE
  quantidade > 0 and
  valor_unitario > 0 and
  custo_unitario > 0 and
  ano_venda BETWEEN 2024 and 2025
GROUP BY 
  mes, ano
ORDER BY 
  ano_venda, mes_venda

### 📅 Questão 2: Sazonalidade e Periodicidade

#### Quais períodos apresentam melhor e pior performance?

**Análises Disponíveis:**

1. **Semestral** (2024-2025):
   * Comparação 1º vs 2º semestre
   * Identifica padrões semestrais

2. **Mensal** (2024-2025):
   * Performance mês a mês
   * Identifica melhores/piores meses

3. **Quinzenal** (2024):
   * Granularidade: 1ª vs 2ª quinzena do mês
   * Identifica padrões intra-mensais

In [0]:
SELECT 
  ano_venda,
  CASE
    WHEN mes_venda <= 6 
      THEN '1º Semestre'
    WHEN mes_venda > 6 
      THEN '2º Semestre'
  END as semestre,
  SUM(faturamento) AS faturamento,
  SUM(lucro) AS lucro,
  SUM(quantidade) AS quantidade_pecas
FROM 
  tb_fato_vendas fv
JOIN 
  tb_dim_tempo ddt ON fv.sk_data_venda = ddt.sk_data_venda
WHERE
  fv.quantidade IS NOT NULL and
  fv.quantidade > 0 and
  fv.valor_unitario > 0 and
  fv.custo_unitario > 0 AND
  ano_venda BETWEEN 2024 and 2025
GROUP BY 
  ano_venda, semestre
ORDER BY 
  ano_venda

In [0]:
SELECT 
  ano_venda,
  CASE
    WHEN mes_venda <= 6 
      THEN '1º Semestre'
    WHEN mes_venda > 6 
      THEN '2º Semestre'
  END as semestre,
  SUM(faturamento) AS faturamento,
  SUM(lucro) AS lucro,
  SUM(quantidade) AS quantidade_pecas
FROM 
  tb_bench
WHERE
  quantidade IS NOT NULL and
  quantidade > 0 and
  valor_unitario > 0 and
  custo_unitario > 0
GROUP BY 
  ano_venda, semestre
ORDER BY 
  ano_venda

In [0]:
SELECT 
    ano_venda, 
    mes_venda, 
    SUM(faturamento) AS faturamento, 
    SUM(lucro) as lucro,
    SUM(quantidade) AS quantidade_pecas
FROM 
  tb_fato_vendas fv
JOIN 
  tb_dim_data_venda ddt on fv.sk_data_venda = ddt.sk_data_venda
WHERE
  fv.quantidade IS NOT NULL and
  fv.quantidade > 0 and
  fv.valor_unitario > 0 and
  fv.custo_unitario > 0
GROUP BY  
  ano_venda, mes_venda
ORDER BY 
  ano_venda, mes_venda

In [0]:
SELECT 
  ano_venda,
  mes_venda,
  CASE
    WHEN day(data_venda) <= 15 
      THEN '1º Quinzena'
    WHEN day(data_venda) > 15 
      THEN '2º Quinzena'
  END AS Semestre,
  SUM(faturamento) AS faturamento,
  SUM(lucro) AS lucro,
  SUM(quantidade) AS quantidade_pecas
FROM 
  tb_fato_vendas fv
JOIN 
  tb_dim_data_venda ddt ON fv.sk_data_venda = ddt.sk_data_venda
WHERE
  fv.quantidade IS NOT NULL and
  fv.quantidade > 0 and
  fv.valor_unitario > 0 and
  fv.custo_unitario > 0
GROUP BY 
  ano_venda, mes_venda, semestre
ORDER BY 
  ano_venda,mes_venda

### 🏪 Questão 3: Desempenho por Localidade

#### Qual o desempenho das vendas por loja, grupo de lojas e vendedor?

**Métricas de Desempenho:**
* Faturamento total
* Lucro total
* Margem de lucro (%)
* Ticket médio (faturamento / nº vendas)
* Lucro médio unitário

**Análises Disponíveis:**

1. **Por Grupo de Loja e Loja Individual**:
   * Hierarquia: Grupo → Loja
   * Ranking de performance
   * Identifica melhores/piores unidades

2. **Por Vendedor**:
   * Performance individual
   * Rankings de vendedores
   * Base para comissões e incentivos

In [0]:
SELECT 
  grupo_loja,
  loja_nome, 
  SUM(faturamento) AS faturamento, 
  SUM(lucro) AS lucro,
  SUM(lucro) / SUM(faturamento) * 100 AS margem_lucro,
  SUM(faturamento) / count(IdNotaFiscal) AS ticket_medio,
  SUM(lucro) / SUM(quantidade) AS lucro_medio_unitario
FROM 
  tb_fato_vendas fv
JOIN 
  tb_dim_loja dl ON fv.sk_loja = dl.sk_loja
WHERE
  fv.quantidade IS NOT NULL and
  fv.quantidade > 0 and
  fv.valor_unitario > 0 and
  fv.custo_unitario > 0
GROUP BY 
   grupo_loja, loja_nome
ORDER BY 
  faturamento desc

In [0]:
SELECT
  vendedor_nome,
  SUM(faturamento) AS faturamento,
  SUM(lucro)AS lucro,
  SUM(lucro) / SUM(faturamento) * 100 AS margem_lucro,
  SUM(faturamento) / count(IdNotaFiscal) AS ticket_medio,
  SUM(lucro) / SUM(quantidade) AS lucro_medio_unitario
FROM 
  tb_fato_vendas fv
JOIN 
  tb_dim_vendedor dv ON fv.sk_vendedor = dv.sk_vendedor
WHERE
  fv.quantidade IS NOT NULL and
  fv.quantidade > 0 and
  fv.valor_unitario > 0 and
  fv.custo_unitario > 0  and
  dv.sk_vendedor = fv.sk_vendedor

GROUP BY 
  vendedor_nome
ORDER BY 
  faturamento DESC
   

### 🔁 Questão 4: Comparação entre Anos

#### Como está o comparativo entre anos?

**Métricas Comparadas (2024 vs 2025):**
* Faturamento total
* Lucro total
* Custo total
* Quantidade de peças vendidas
* Número de vendas

In [0]:
SELECT
    ano_venda AS ano,
    SUM(faturamento) AS faturamento, 
    SUM(lucro) AS lucro,  
    SUM(custo_unitario * quantidade) AS custo_total,
    SUM(quantidade) AS qtd_pecas_vendidas,
    COUNT(IdNotaFiscal) AS n_vendas,
    ROW_NUMBER() OVER(
      PARTITION BY ano_venda
      ORDER BY sum(faturamento) DESC
  ) AS rn_quantidade
  FROM 
    tb_fato_vendas fv
  JOIN 
    tb_dim_data_venda ddt ON ddt.sk_data_venda = fv.sk_data_venda
  JOIN
    tb_dim_produtos dp ON dp.sk_produto = fv.sk_produto
  WHERE
    fv.quantidade IS NOT NULL and
    fv.quantidade > 0 and
    fv.valor_unitario > 0 and
    fv.custo_unitario > 0
  GROUP BY  ano

### 📊 Questão 5: Concentração de Volume

#### Quais unidades e grupos concentram maior volume de vendas?

**Objetivo:**
Identificar onde está concentrado o **volume de transações** (número de vendas) versus o **volume de produtos** (quantidade vendida).

**Métricas:**
* Contagem de vendas por grupo/loja
* Quantidade de peças vendidas
* Distribuição percentual


In [0]:
SELECT 
  grupo_loja,
  loja_nome,
  count(IdNotaFiscal) AS n_vendas,
  SUM(quantidade) AS qtd_vendida
FROM
  tb_fato_vendas fv
JOIN 
  tb_dim_loja dl ON dl.sk_loja = fv.sk_loja
WHERE
  fv.quantidade IS NOT NULL and
  fv.quantidade > 0 and
  fv.valor_unitario > 0 and
  fv.custo_unitario > 0  
GROUP BY 
   grupo_loja, loja_nome
ORDER BY n_vendas DESC


In [0]:
SELECT 
  grupo_loja,
  count(IdNotaFiscal) AS n_vendas,
  SUM(quantidade) AS qtd_vendida
FROM
  tb_fato_vendas fv
JOIN 
  tb_dim_loja dl ON dl.sk_loja = fv.sk_loja
WHERE
  fv.quantidade IS NOT NULL and
  fv.quantidade > 0 and
  fv.valor_unitario > 0 and
  fv.custo_unitario > 0
GROUP BY 
  grupo_loja
ORDER BY n_vendas DESC